In [1]:

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor, Pool
import pandas as pd
import numpy as np
from flask import Flask
from flask_restx import Api, Resource, fields
import joblib

In [2]:
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')

dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv',index_col=0)

print(dataTraining.shape)
print(dataTesting.shape)
dataTraining.head()

(79800, 21)
(34200, 19)


,Unnamed: 0,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity
0,0,7hUhmkALyQ8SX9mJs5XI3D,Love and Rockets,Love and Rockets,Motorcycle,211533,False,0.305,0.8490,9,...,1,0.0549,0.000058,0.056700,0.4640,0.3200,141.793,4,goth,22
1,1,5x59U89ZnjZXuNAAlc8X1u,Filippa Giordano,Filippa Giordano,"Addio del passato - From ""La traviata""",196000,False,0.287,0.1900,7,...,0,0.0370,0.930000,0.000356,0.0834,0.1330,83.685,4,opera,22
2,2,70Vng5jLzoJLmeLu3ayBQq,Susumu Yokota,Symbol,Purple Rose Minuet,216506,False,0.583,0.5090,1,...,1,0.0362,0.777000,0.202000,0.1150,0.5440,90.459,3,idm,37
3,3,1cRfzLJapgtwJ61xszs37b,Franz Liszt;YUNDI,Relajación y siestas,"Liebeslied (Widmung), S. 566",218346,False,0.163,0.0368,8,...,1,0.0472,0.991000,0.899000,0.1070,0.0387,69.442,3,classical,0
4,4,47d5lYjbiMy0EdMRV8lRou,Scooter,Scooter Forever,The Darkside,173160,False,0.647,0.9210,2,...,1,0.1850,0.000939,0.371000,0.1310,0.1710,137.981,4,techno,27


In [3]:
# duration en minutos
dataTraining['duration_min'] = dataTraining['duration_ms'] / 60000
dataTesting['duration_min'] = dataTesting['duration_ms'] / 60000

print(dataTraining.shape)

(79800, 22)


In [4]:
y_full = dataTraining['popularity']
X_full = dataTraining.drop(columns=['popularity', 'Unnamed: 0'])

X_train, X_val, y_train, y_val = train_test_split(X_full,y_full,test_size=0.2,random_state=42)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(63840, 20) (15960, 20) (63840,) (15960,)


In [5]:

# TARGET ENCODING con folds

kf = KFold(n_splits=5, shuffle=True, random_state=13)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
dataTesting = dataTesting.reset_index(drop=True)

# Nuevas variables de target encoding 
te_cols = ['artist_pop_mean','artist_pop_max','album_pop_mean','album_pop_std','genre_pop_mean','genre_pop_std']

X_train_oof = X_train.copy()

for col in te_cols:
    X_train_oof[col] = np.nan


for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):

    X_fold_train = X_train.iloc[train_idx].copy()
    y_fold_train = y_train.iloc[train_idx].copy()
    X_fold_val = X_train.iloc[val_idx].copy()

    df_temp = X_fold_train.copy()
    df_temp['popularity'] = y_fold_train.values

    global_mean = y_fold_train.mean()


    # ARTISTA: mean y max
    artist_means = df_temp.groupby('artists')['popularity'].mean()
    artist_maxs = df_temp.groupby('artists')['popularity'].max()

    X_train_oof.loc[val_idx, 'artist_pop_mean'] = (
        X_fold_val['artists'].map(artist_means).fillna(global_mean).values
    )
    X_train_oof.loc[val_idx, 'artist_pop_max'] = (
        X_fold_val['artists'].map(artist_maxs).fillna(global_mean).values
    )


    # ÁLBUM: mean y std
    album_means = df_temp.groupby('album_name')['popularity'].mean()
    album_stds = df_temp.groupby('album_name')['popularity'].std().fillna(0)

    X_train_oof.loc[val_idx, 'album_pop_mean'] = (
        X_fold_val['album_name'].map(album_means).fillna(global_mean).values
    )
    X_train_oof.loc[val_idx, 'album_pop_std'] = (
        X_fold_val['album_name'].map(album_stds).fillna(0).values
    )


    # GÉNERO: mean y std
    genre_means = df_temp.groupby('track_genre')['popularity'].mean()
    genre_stds = df_temp.groupby('track_genre')['popularity'].std().fillna(0)

    X_train_oof.loc[val_idx, 'genre_pop_mean'] = (
        X_fold_val['track_genre'].map(genre_means).fillna(global_mean).values
    )
    X_train_oof.loc[val_idx, 'genre_pop_std'] = (
        X_fold_val['track_genre'].map(genre_stds).fillna(0).values
    )


In [6]:

# ENCODING FINAL (VAL Y TEST)- se usa valores de train para no juntar datasets

df_temp = X_train.copy()
df_temp['popularity'] = y_train.values
global_mean = y_train.mean()


# Calculamos los promedios usando todo X_train
artist_means = df_temp.groupby('artists')['popularity'].mean()
artist_maxs  = df_temp.groupby('artists')['popularity'].max()

album_means = df_temp.groupby('album_name')['popularity'].mean()
album_stds  = df_temp.groupby('album_name')['popularity'].std().fillna(0)

genre_means = df_temp.groupby('track_genre')['popularity'].mean()
genre_stds  = df_temp.groupby('track_genre')['popularity'].std().fillna(0)


# Aplicamos a VALIDACIÓN
X_va_e = X_val.copy()
X_va_e['artist_pop_mean'] = X_va_e['artists'].map(artist_means).fillna(global_mean)
X_va_e['artist_pop_max']  = X_va_e['artists'].map(artist_maxs).fillna(global_mean)
X_va_e['album_pop_mean']  = X_va_e['album_name'].map(album_means).fillna(global_mean)
X_va_e['album_pop_std']   = X_va_e['album_name'].map(album_stds).fillna(0)
X_va_e['genre_pop_mean']  = X_va_e['track_genre'].map(genre_means).fillna(global_mean)
X_va_e['genre_pop_std']   = X_va_e['track_genre'].map(genre_stds).fillna(0)


# Aplicamos a TESTING
X_te_e = dataTesting.copy()
X_te_e['artist_pop_mean'] = X_te_e['artists'].map(artist_means).fillna(global_mean)
X_te_e['artist_pop_max']  = X_te_e['artists'].map(artist_maxs).fillna(global_mean)
X_te_e['album_pop_mean']  = X_te_e['album_name'].map(album_means).fillna(global_mean)
X_te_e['album_pop_std']   = X_te_e['album_name'].map(album_stds).fillna(0)
X_te_e['genre_pop_mean']  = X_te_e['track_genre'].map(genre_means).fillna(global_mean)
X_te_e['genre_pop_std']   = X_te_e['track_genre'].map(genre_stds).fillna(0)


print("Encoding final listo")
print(X_train_oof.shape, X_va_e.shape, X_te_e.shape)

Encoding final listo
(63840, 26) (15960, 26) (34200, 26)


In [7]:
DROP_COLS = ['track_id','artists','album_name','track_name','duration_ms','track_genre']

for df in (X_train_oof, X_va_e, X_te_e):
    for col in DROP_COLS:
        if col in df.columns:
            df.drop(columns=col, inplace=True)



cat_features = ['key','mode','time_signature','explicit',]
cat_features = [col for col in cat_features if col in X_train_oof.columns]


# Convertir categóricas a string
for col in cat_features:
    X_train_oof[col] = X_train_oof[col].astype(str).fillna('NaN')
    X_va_e[col] = X_va_e[col].astype(str).fillna('NaN')
    X_te_e[col] = X_te_e[col].astype(str).fillna('NaN')


# Rellenar NaN numéricos
for df in (X_train_oof, X_va_e, X_te_e):
    for col in df.columns:
        if col not in cat_features and df[col].isna().any():
            df[col] = df[col].fillna(0)


# Asegurar mismo orden de columnas
X_va_e = X_va_e[X_train_oof.columns]
X_te_e = X_te_e[X_train_oof.columns]


print(f"X_train_oof: {X_train_oof.shape}")
print(f"X_va_e: {X_va_e.shape}")
print(f"X_te_e: {X_te_e.shape}")
print(f"\ncat_features ({len(cat_features)}): {cat_features}")

X_train_oof: (63840, 20)
X_va_e: (15960, 20)
X_te_e: (34200, 20)

cat_features (4): ['key', 'mode', 'time_signature', 'explicit']


In [8]:
X_train_oof.columns

Index(['explicit', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'duration_min', 'artist_pop_mean',
       'artist_pop_max', 'album_pop_mean', 'album_pop_std', 'genre_pop_mean',
       'genre_pop_std'],
      dtype='object')

In [9]:
modelo_cat = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.09,
    depth=8,
    l2_leaf_reg=5,
    random_strength=5,
    bagging_temperature=0.5,
    random_seed=13,
    loss_function='RMSE',
    eval_metric='RMSE',
    early_stopping_rounds=80,
    verbose=100
)

modelo_cat.fit(
    X_train_oof,
    y_train,
    cat_features=cat_features,
    eval_set=(X_va_e, y_val),
    use_best_model=True
)

pred_va = modelo_cat.predict(X_va_e)
pred_va = np.clip(pred_va, 0, 100)

rmse = np.sqrt(mean_squared_error(y_val, pred_va))
mae = mean_absolute_error(y_val, pred_va)


print("RMSE:", rmse)
print("MAE:", mae)


0:	learn: 20.8711593	test: 20.7365600	best: 20.7365600 (0)	total: 190ms	remaining: 4m 44s
100:	learn: 8.5738329	test: 8.3658638	best: 8.3658638 (100)	total: 5.45s	remaining: 1m 15s
200:	learn: 8.0574802	test: 7.9988632	best: 7.9988632 (200)	total: 10.8s	remaining: 1m 9s
300:	learn: 7.7787911	test: 7.8947394	best: 7.8946407 (298)	total: 16.2s	remaining: 1m 4s
400:	learn: 7.5755944	test: 7.8350620	best: 7.8350620 (400)	total: 21.5s	remaining: 58.9s
500:	learn: 7.3914781	test: 7.7999723	best: 7.7998185 (499)	total: 26.8s	remaining: 53.5s
600:	learn: 7.2383384	test: 7.7682541	best: 7.7682541 (600)	total: 32.3s	remaining: 48.4s
700:	learn: 7.0891494	test: 7.7454438	best: 7.7451370 (687)	total: 38.2s	remaining: 43.6s
800:	learn: 6.9561899	test: 7.7289580	best: 7.7287942 (796)	total: 44.5s	remaining: 38.8s
900:	learn: 6.8261216	test: 7.7193108	best: 7.7183913 (885)	total: 50.5s	remaining: 33.6s
1000:	learn: 6.7068286	test: 7.7136798	best: 7.7126819 (971)	total: 56.2s	remaining: 28s
1100:	lear

In [10]:
df_temp = X_train.copy()
df_temp['popularity']   = y_train.values
df_temp['artists_norm'] = df_temp['artists'].str.strip().str.lower()
df_temp['album_norm']   = df_temp['album_name'].str.strip().str.lower()
df_temp['genre_norm']   = df_temp['track_genre'].str.strip().str.lower()

global_mean = y_train.mean()
artist_means = df_temp.groupby('artists_norm')['popularity'].mean().to_dict()
artist_maxs  = df_temp.groupby('artists_norm')['popularity'].max().to_dict()
album_means  = df_temp.groupby('album_norm')['popularity'].mean().to_dict()
album_stds   = df_temp.groupby('album_norm')['popularity'].std().fillna(0).to_dict()
genre_means  = df_temp.groupby('genre_norm')['popularity'].mean().to_dict()
genre_stds   = df_temp.groupby('genre_norm')['popularity'].std().fillna(0).to_dict()

joblib.dump(modelo_cat,   'pop_song_regressor.pkl', compress=3)
joblib.dump(artist_means, 'artista_means.pkl', compress=3)
joblib.dump(artist_maxs,  'artista_maxs.pkl',  compress=3)
joblib.dump(album_means,  'album_means.pkl',  compress=3)
joblib.dump(album_stds,   'album_stds.pkl',   compress=3)
joblib.dump(genre_means,  'genero_means.pkl',  compress=3)
joblib.dump(genre_stds,   'genero_stds.pkl',   compress=3)
joblib.dump(global_mean,  'global_mean.pkl',  compress=3)

variables_modelo = list(X_train_oof.columns)
joblib.dump(variables_modelo, 'variables_modelo.pkl', compress=3)

generos_disponibles = sorted(df_temp['track_genre'].unique().tolist())
joblib.dump(generos_disponibles, 'generos_disponibles.pkl', compress=3)

print("Archivos guardados")

Archivos guardados


In [11]:
app = Flask(__name__)

api = Api(
    app,
    version='1.0',
    title='Song Popularity API',
    description='Song Popularity Prediction API'
)

ns = api.namespace('predict', description='Song Popularity Regressor')

modelo = joblib.load('pop_song_regressor.pkl')
artist_means = joblib.load('artista_means.pkl')
artist_maxs  = joblib.load('artista_maxs.pkl')
album_means  = joblib.load('album_means.pkl')
album_stds   = joblib.load('album_stds.pkl')
genre_means  = joblib.load('genero_means.pkl')
genre_stds   = joblib.load('genero_stds.pkl')
global_mean  = joblib.load('global_mean.pkl')
variables_modelo    = joblib.load('variables_modelo.pkl')
generos_disponibles = joblib.load('generos_disponibles.pkl')

parser = ns.parser()
parser.add_argument('artists',          type=str,   required=True, help='Nombre(s) del/los artista(s). Si son varios separar con ;', location='args')
parser.add_argument('album_name',       type=str,   required=True, help='Nombre del album', location='args')
parser.add_argument('track_genre',      type=str,   required=True, choices=generos_disponibles, help='Genero de la cancion', location='args')
parser.add_argument('duration_min',     type=float, required=True, help='Duracion de la cancion en minutos', location='args')
parser.add_argument('tempo',            type=float, required=True, help='Tempo en BPM', location='args')
parser.add_argument('loudness',         type=float, required=True, help='Loudness en dB', location='args')
parser.add_argument('acousticness',     type=float, required=True, help='Acousticness (0-1)', location='args')
parser.add_argument('instrumentalness', type=float, required=True, help='Instrumentalness (0-1)', location='args')
parser.add_argument('energy',           type=float, required=True, help='Energy (0-1)', location='args')
parser.add_argument('valence',          type=float, required=True, help='Valence (0-1)', location='args')
parser.add_argument('speechiness',      type=float, required=True, help='Speechiness (0-1)', location='args')
parser.add_argument('liveness',         type=float, required=True, help='Liveness (0-1)', location='args')
parser.add_argument('danceability',     type=float, required=True, help='Danceability (0-1)', location='args')
parser.add_argument('key',              type=int,   required=True, help='Key de la cancion (-1 a 11)', location='args')
parser.add_argument('mode',             type=int,   required=True, help='Mode (0 menor, 1 mayor)', location='args')
parser.add_argument('time_signature',   type=int,   required=True, help='Time signature (3-7)', location='args')
parser.add_argument('explicit',         type=str,   required=True, choices=['True', 'False'], help='Si la cancion es explicita', location='args')


resource_fields = api.model('Resource', {
    'result': fields.Float,
})

def crear_df_como_entrenamiento(args):
    artists     = args['artists'].strip().lower()
    album_name  = args['album_name'].strip().lower()
    track_genre = args['track_genre'].strip().lower()
    artist_pop_mean = artist_means.get(artists, global_mean)
    artist_pop_max  = artist_maxs.get(artists, global_mean)
    album_pop_mean  = album_means.get(album_name, artist_pop_mean)
    album_pop_std   = album_stds.get(album_name, 0)
    genre_pop_mean  = genre_means.get(track_genre, global_mean)
    genre_pop_std   = genre_stds.get(track_genre, 0)

    row = {
        'explicit':         args['explicit'],
        'danceability':     args['danceability'],
        'energy':           args['energy'],
        'key':              str(args['key']),
        'loudness':         args['loudness'],
        'mode':             str(args['mode']),
        'speechiness':      args['speechiness'],
        'acousticness':     args['acousticness'],
        'instrumentalness': args['instrumentalness'],
        'liveness':         args['liveness'],
        'valence':          args['valence'],
        'tempo':            args['tempo'],
        'time_signature':   str(args['time_signature']),
        'duration_min':     args['duration_min'],
        'artist_pop_mean':  artist_pop_mean,
        'artist_pop_max':   artist_pop_max,
        'album_pop_mean':   album_pop_mean,
        'album_pop_std':    album_pop_std,
        'genre_pop_mean':   genre_pop_mean,
        'genre_pop_std':    genre_pop_std,
    }

    return pd.DataFrame([row], columns=variables_modelo)

In [12]:

@ns.route('/')
class SongPopularityApi(Resource):

    @ns.expect(parser)
    @ns.marshal_with(resource_fields)
    def get(self):
        args = parser.parse_args()
        df_ingresado = crear_df_como_entrenamiento(args)

        prediccion = modelo.predict(df_ingresado)[0]
        return {'result': float(prediccion)}, 200


if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.80.21:5000
Press CTRL+C to quit
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET /swaggerui/droid-sans.css HTTP/1.1" 304 -
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET /swaggerui/swagger-ui-bundle.js HTTP/1.1" 304 -
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET /swaggerui/swagger-ui.css HTTP/1.1" 304 -
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET /swaggerui/swagger-ui-standalone-preset.js HTTP/1.1" 304 -
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET /swagger.json HTTP/1.1" 200 -
127.0.0.1 - - [26/Apr/2026 11:12:46] "GET /swaggerui/favicon-32x32.png HTTP/1.1" 304 -
127.0.0.1 - - [26/Apr/2026 11:13:43] "GET /predict/?artists=Bad%20Bunny&album_name=Un%20Verano%20Sin%20Ti&track_genre=reggaeton&duration_min=3.5&tempo=95&loudness=-5.5&acousticness=0.15&instrumentalness=0&energy=0.78&valence=0.65&speechiness=0.08&liveness=0.12&danceability=0.82&key=5&mo